In [33]:
import openai
import os
import re
from IPython.display import display, Markdown

# 1. 初始化客户端 (请替换为你自己的真实凭证)
client = openai.OpenAI(
    api_key=os.environ["LLM_API_KEY"], 
    base_url="https://new.12ai.org/v1"
)

# 目标保存路径 (使用 r 前缀处理 Windows 路径)
save_directory = r"..\Judge_Input\Therm._Deg"
file_name = "gemini3.1pro-3.txt"
full_path = os.path.join(save_directory, file_name)

# =========================
# 2. 定义提示词
# =========================

user_prompt = (
    "Design a biodegradable polyester with improved heat resistance but without excessively suppressing degradation kinetics."
)

# =========================
# 3. 请求模型
# =========================

print("正在生成设计方案...")

response = client.chat.completions.create(
    model="gemini-3.1-pro-preview",
    messages=[
        {"role": "user", "content": user_prompt}
    ]
)

# =========================
# 4. 兼容不同 API 返回格式
# =========================

def extract_content(response):
    """
    兼容三类返回：
    1. 标准 OpenAI ChatCompletion 对象
    2. dict 类型返回
    3. 第三方 API 直接返回 str
    """

    # 情况 1：第三方 API 直接返回字符串
    if isinstance(response, str):
        return response

    # 情况 2：dict 类型
    if isinstance(response, dict):
        try:
            return response["choices"][0]["message"]["content"]
        except Exception:
            return json.dumps(response, ensure_ascii=False, indent=2)

    # 情况 3：标准 OpenAI SDK 对象
    if hasattr(response, "choices"):
        return response.choices[0].message.content

    # 兜底
    return str(response)


raw_content = extract_content(response)

# =========================
# 5. 去除 <think>...</think>
# =========================

clean_content = re.sub(
    r"<think>.*?</think>\s*",
    "",
    raw_content,
    flags=re.DOTALL | re.IGNORECASE
).strip()

# =========================
# 6. 保存结果
# =========================

try:
    os.makedirs(save_directory, exist_ok=True)

    with open(full_path, "w", encoding="utf-8") as f:
        f.write(clean_content)

    print(f"成功保存纯净版结果至: {full_path}")

except Exception as e:
    print(f"保存文件时出错: {e}")

# =========================
# 7. Notebook 预览
# =========================

display(Markdown("### 模型生成的设计方案（已去除思考过程）："))
display(Markdown(clean_content))

正在生成设计方案...
成功保存纯净版结果至: ..\Judge_Input\Therm._Deg\gemini3.1pro-3.txt


### 模型生成的设计方案（已去除思考过程）：

To design a biodegradable polyester that balances high heat resistance with rapid degradation kinetics, we must address the "thermal-stability paradox": features that increase heat resistance (rigidity, high crystallinity, and strong intermolecular forces) typically impede the diffusion of water and enzymes required for degradation.

The following design proposes a **Segmented Co-polyester** utilizing a bio-based bicyclic monomer for rigidity and "weak-link" glycolic acid segments to facilitate degradation.

---

### 1. The Chemical Composition
The proposed polymer is a **Poly(isosorbide-co-succinate-co-glycolate)** terpolymer.

#### The Monomer Toolkit:
1.  **Isosorbide (1,4:3,6-dianhydro-D-glucitol):** A rigid, bio-based bicyclic diol derived from starch. It significantly raises the Glass Transition Temperature ($T_g$) by restricting chain mobility.
2.  **Succinic Acid:** A bio-based aliphatic diacid that provides flexibility and maintains the ester-linkage density.
3.  **Glycolic Acid (or Glycolide):** The smallest $\alpha$-hydroxy acid. It is highly hydrophilic and increases the rate of ester hydrolysis compared to lactic acid or longer chain diacids.

### 2. Structural Architecture: The "Segmented" Approach
Rather than a purely random copolymer, a **multi-block (segmented) architecture** is preferred.

*   **Hard Segments (Heat Resistance):** Blocks of **Isosorbide Succinate**. The bicyclic rings of Isosorbide prevent the chain from softening at low temperatures, pushing the $T_g$ toward $100–120^\circ\text{C}$ (depending on the ratio).
*   **Labile Segments (Degradation Trigger):** Short blocks of **Polyglycolic Acid (PGA)** or random Glycolate insertions.

### 3. Mechanism of Improved Heat Resistance
*   **Chain Rigidity:** Isosorbide’s V-shaped, fused-ring structure imposes high steric hindrance. This prevents the polymer from transitioning into a rubbery state at boiling water temperatures ($100^\circ\text{C}$), a common failure point for standard PLA.
*   **Hydrogen Bonding:** The secondary hydroxyls in Isosorbide, if not fully reacted or when part of the chain end, can form localized hydrogen bonding networks that stabilize the matrix thermally.

### 4. Mechanism of Maintained Degradation
Standard high-$T_g$ polyesters (like PEF or PET) are hydrophobic and exclude water. This design bypasses that via:
*   **Hydrophilicity Tuning:** Glycolic acid is significantly more hydrophilic than lactic acid. Its presence lowers the water contact angle of the bulk polymer, allowing water to permeate the "hard" Isosorbide matrix.
*   **"Zip-per" Hydrolysis:** Once water enters, the glycolic ester bonds—which have higher elective susceptibility to hydrolysis due to less steric hindrance than the Isosorbide-succinate bonds—break first. This fragments the high-molecular-weight chains into smaller oligomers, rapidly increasing the surface area for enzymatic attack.
*   **Reduced Crystallinity (Controlled Amorphousness):** By using Isosorbide (which is inherently asymmetric), we can prevent the formation of large, impenetrable crystalline domains. An amorphous but **rigid** glass degrades faster than a crystalline solid because the entire bulk is accessible to water.

### 5. Predicted Properties
| Property | Value (Target) | Comparison (PLA) |
| :--- | :--- | :--- |
| **Glass Transition ($T_g$)** | $95–115^\circ\text{C}$ | $55–60^\circ\text{C}$ |
| **Melting Point ($T_m$)** | $180–210^\circ\text{C}$ | $150–180^\circ\text{C}$ |
| **HDT (Heat Deflection)** | High (Stable in boiling water) | Low (Softens at $60^\circ\text{C}$) |
| **Degradation Rate** | 6–12 months (Compost) | 6–24 months |

### 6. Synthesis Strategy
The polymer is synthesized via **melt polycondensation**:
1.  **Pre-polymerization:** Esterification of Succinic acid and Isosorbide with a slight excess of diol.
2.  **Chain Extension:** Addition of Glycolide or Glycolic acid under high vacuum with a catalyst (e.g., Zinc lactate or Stannous octoate).
3.  **Tuning:** The $T_g$ can be "dialed in" by adjusting the Isosorbide-to-Succinic acid ratio. If the material becomes too brittle, a small fraction of 1,4-butanediol can be added as a toughening agent.

### Summary
This design solves the heat-resistance problem by using **Isosorbide** to mechanically stiffen the polymer backbone, while ensuring bio-assimilation by incorporating **Glycolic acid** to act as a "chemical fuse" that invites water into the matrix, ensuring that high thermal stability does not result in environmental persistence.